<header>
   <p style='font-size:36px;font-family:Arial; color:#F0F0F0; background-color: #00233c; padding-left: 20pt; padding-top: 20pt;padding-bottom: 10pt; padding-right: 20pt;'>
        Content-Based Collection with Open Table Format (OTF)
  <br>
       <img id="teradata-logo" src="https://storage.googleapis.com/clearscape_analytics_demo_data/DEMO_Logo/teradata.svg" alt="Teradata" style="width: 125px; height: auto; margin-top: 20pt;">
    </p>
</header>

## Introduction

This notebook demonstrates how to create a **Content-Based Collection** using tables from an **Open Table Format (OTF)** datalake

### Example table used in this notebook

| Parameter | Example value |
|---|---|
| Datalake name | `datalake_iceberg_glue` |
| Database name | `tdotf_test_company_db` |
| Table name | `tdotf_test_assets` |
| Data column | `location` |
| Key column | `asset_id` |

You will be prompted to enter your own values when you run the notebook.

### What You'll Learn

1. Connecting to Vantage and authenticating with the Collection service
2. Referencing an OTF table as a teradataml DataFrame
3. Creating a content-based collection from an OTF table
4. Checking collection status and details
5. Running a similarity search against the OTF-backed collection
6. Cleaning up resources

---
## 1. Setup

### Install teradatagenai

In [ ]:
# %pip install teradatagenai --upgrade

### Import Required Modules

In [ ]:
from getpass import getpass

from teradatagenai import Collection, TeradataAI, ContentBasedIndex, ColumnInfo
from teradataml import create_context, DataFrame, in_schema, set_auth_token, remove_context

### Connect to Vantage

In [ ]:
hostname = getpass(prompt='Hostname: ')
username = getpass(prompt='Username: ')
password = getpass(prompt='Password: ')
create_context(host=hostname, username=username, password=password)
print("Connected to Vantage")

### Authenticate with the Collection Service

In [ ]:
base_url = getpass(prompt='Base URL (e.g. https://<host>): ')
set_auth_token(base_url=base_url, username=username, password=password)
print("Authentication token set")

---
## 2. Specify OTF Table Details

Enter the three-part identifier for your OTF table.

> **Example values used during development:**  
> - Datalake name : `datalake_iceberg_glue`  
> - Database name : `tdotf_test_company_db`  
> - Table name    : `tdotf_test_assets`  
> - Data column   : `location`  
> - Key column    : `asset_id`

In [ ]:
datalake_name = getpass(prompt='Datalake name: ')
database_name = getpass(prompt='Database name: ')
table_name    = getpass(prompt='Table name: ')
data_col      = input('Data column (text column to embed): ')
key_col       = input('Key column (unique identifier): ')

---
## 3. Create a DataFrame from the OTF Table

Use `in_schema(database, table, datalake)` to reference an OTF table.  

In [ ]:
otf_table = in_schema(database_name, table_name, datalake_name)
otf_df = DataFrame(otf_table)

otf_df.head(1)

asset_id,asset_tag,asset_type,owner_employee_id,purchase_date,warranty_until,purchase_price,location,status,last_maintenance,serial_number,description
1,ASSET-LAP-001,Laptop,33333,22/06/15,25/06/15,1450.000,San Diego - 5th Floor - Dev,In Service,24/08/01,SN-LAP-20220615-001,b'123ABC1230...'


---
## 4. Initialize the Embedding Model

In [5]:
# Using Teradata-hosted embedding model — no extra credentials needed.
# Replace with your preferred model if needed.
embedding_model = TeradataAI(
    api_type="td_hosted",
    model_name="amazon.titan-embed-text-v2:0"
)
print("Embedding model initialized")

Embedding model initialized


---
## 5. Create the Content-Based Collection

Pass the OTF DataFrame directly as `object_names` in a `ContentBasedIndex`.  
No special handling is required — the Collection service receives the qualified OTF table name automatically.

In [12]:
key_column_info = ColumnInfo(
    name=key_col,
    description=f"Primary key: {key_col}"
)

data_column_info = ColumnInfo(
    name=data_col,
    description=f"Data column to embed: {data_col}"
)

content_index = ContentBasedIndex(
    object_names=otf_df,
    key_columns=[key_column_info],
    data_columns=[data_column_info]
)
print("ContentBasedIndex created")

ContentBasedIndex created


In [13]:
collection_name = input('Collection name to create: ')
coll = Collection(name=collection_name)
coll.create(
    index=content_index,
    embedding_model=embedding_model
)
print(f"Collection '{collection_name}' creation triggered")

/root/sm/LCL/lcl-NexusAI/teradatagenai/vector_store/collection.py:711: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Collection create request is accepted and in progress
Collection 'otf_test_collection' creation triggered


---
## 6. Check Status and Details

In [15]:
coll.status()

collection_name,collection_status
otf_test_collection,ready


In [16]:
coll.get_details(elaborate=True, return_type="json")

{'collection_name': 'otf_test_collection',
 'collection_status': 'ready',
 'collection_type': 'CONTENT-BASED',
 'collection_description': None,
 'target_database': 'OTF_USER',
 'permission': 'ADMIN',
 'last_updated': '2026-07-22 05:18:34.848609+00:00',
 'error_message': None,
 'embedding_datatype': 'VECTOR32',
 'use_simd': False,
 'dynamic_update': False,
 'object_names': ['"datalake_iceberg_glue"."tdotf_test_company_db"."tdotf_test_assets"'],
 'key_columns': {'"datalake_iceberg_glue"."tdotf_test_company_db"."tdotf_test_assets"': ['"asset_id"']},
 'key_columns_info': [{'name': '"asset_id"',
   'datatype': 'INTEGER',
   'description': 'Primary key: asset_id'}],
 'metadata_columns': None,
 'metadata_columns_info': None,
 'data_columns': {'"datalake_iceberg_glue"."tdotf_test_company_db"."tdotf_test_assets"': ['"location"']},
 'data_columns_info': [{'name': '"location"',
   'datatype': 'VARCHAR(2000) CHAR SET UNICODE',
   'description': 'Data column to embed: location'}],
 'image_column': 

---
## 7. Similarity Search

In [ ]:
question = input('Enter your search question: ')

results = coll.similarity_search(
    question=question,
    top_k=5
)
print(results)

similar_objects_count:2
similar_objects:
      score                                   DataBaseName          TableName  TD_ID  asset_id                       location  index_label
0  0.286351  datalake_iceberg_glue"."tdotf_test_company_db  tdotf_test_assets      2         1    San Diego - 5th Floor - Dev            1
1  0.523757  datalake_iceberg_glue"."tdotf_test_company_db  tdotf_test_assets      1         2  Los Angeles - 3rd Floor - Ops            0)


---
## 8. Cleanup

In [18]:
coll.destroy()
print(f"Collection '{collection_name}' destroyed")

Collection destroy request is accepted and in progress
Collection 'otf_test_collection' destroyed


In [19]:
remove_context()
print("Database connection removed")

Database connection removed


---
## Summary

In this notebook, you learned how to:

✅ Reference an OTF table using `in_schema(database, table, datalake)`  
✅ Create a content-based collection from an OTF-backed DataFrame  
✅ Run a similarity search against the OTF collection  
✅ Clean up resources  

No special configuration is needed beyond using `in_schema` with three arguments — the teradataml DataFrame carries the correct qualified name automatically.